# 提示词版本管理与回滚

假设你是一名产品经理，你的产品支持系统用 LLM 把新到的工单路由给相应的团队。你想调整路由提示词，让更多与 API 相关的工单流向平台团队。以前，提示词存在于你的代码库中。修改它需要一次 PR、一次 CI 运行和一次部署。回滚也需要同样的改动。Managed Agents 把提示词保存在服务端。每次 `agents.update` 都会产生一个新的不可变版本，会话通过 ID 选择使用哪个版本。你仍然要评审和批准提示词改动，但你批准的是配置里的版本号，而不是代码差异，并且正在运行的服务无需重新构建就能采用该改动。如果出了问题，只需让调用方重新指向旧版本即可完成回滚。

在本 Cookbook 中，我们会创建一个支持工单分诊智能体，更新系统提示词，并在性能下降时把它回滚。

读完本文你将拥有：
- 创建了一个智能体，并看到它返回 `version: 1`
- 针对一组带标注的测试集为某个特定版本打分
- 发布了 v2 提示词，并看到版本号发生移动
- 在没有部署的情况下把出现退化的智能体回滚到 v1

## 前置条件

**必需：**
- Python 3.11+
- `anthropic>=0.91.0`
- 一个 Anthropic API 密钥
- 本目录中的 `example_data/prompt_versioning_and_rollback/support_tickets.jsonl` 测试数据


## 准备


In [1]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv

In [2]:
import json
import os
import time
from collections import defaultdict
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY in your environment or .env file")

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
client = Anthropic()

## 创建智能体（版本 1）

我们先创建一个 Environment（智能体运行所在的容器模板）以及智能体本身。系统提示词很短：把每张工单分类到某个团队和优先级，并且只以 JSON 响应。


In [3]:
env = client.beta.environments.create(name="ticket-triage-env")
ENV_ID = env.id

V1_SYSTEM = """You are a support-ticket triage agent for a usage-billed API product.
Read the ticket and respond with ONLY a single line of raw JSON (no code fences, no prose):
{"team": "<billing|auth|api-platform|dashboard>", "priority": "<P1|P2|P3>"}
Route based on the customer's actual problem, not surface keywords."""

agent = client.beta.agents.create(
    name="ticket-triage",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-prompt-versioning-and-rollback"},
    model=MODEL,
    system=V1_SYSTEM,
)
AGENT_ID = agent.id
print(f"env={ENV_ID}  agent={AGENT_ID} v{agent.version}")

env=env_017FKqTdU6dFgEe6zbbVQvE5  agent=agent_011CZo6t9Cr6Eg5TeFFPWPpi v1


`agents.create` 未经我们要求就返回了 `version: 1`。我们之后的每次更新都会产生一个具有相同 `id` 的新版本，而这个编号正是我们用来固定会话和回滚的依据。


## 载入带标注的测试集

下面有二十张工单，每个团队五张，正确的路由结果已经标注好。这些就是我们要据以打分的基准答案。


In [4]:
fixture = Path("example_data/prompt_versioning_and_rollback/support_tickets.jsonl")
tickets = [json.loads(line) for line in fixture.read_text().splitlines()]
teams = sorted({t["team"] for t in tickets})
print(f"{len(tickets)} tickets across {len(teams)} teams: {teams}")

20 tickets across 4 teams: ['api-platform', 'auth', 'billing', 'dashboard']


## 为版本 1 打分

我们把每张工单都跑一遍 v1，看看它的表现如何。下面的 `triage` 辅助函数会打开一个固定到特定版本的会话，发送一张工单，轮询 `events.list` 直到会话变为空闲，并从 `agent.message` 事件中解析出 JSON 判定结果。

对本 Cookbook 而言关键的部分是 `sessions.create` 上的 `agent=` 参数。传入普通字符串（`agent=AGENT_ID`）会得到当前最新版本。传入 `{"type": "agent", "id": ..., "version": ...}` 则固定到某个确切版本，这正是受控对比所需要的。


In [5]:
def triage(version: int, ticket: dict) -> dict:
    """Run one ticket through a pinned agent version and return its verdict."""
    session = client.beta.sessions.create(
        agent={"type": "agent", "id": AGENT_ID, "version": version},
        environment_id=ENV_ID,
    )
    try:
        prompt = "Subject: " + ticket["subject"] + "\n\n" + ticket["body"]
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": prompt}]}],
        )
        deadline = time.time() + 60
        while time.time() < deadline:
            events = client.beta.sessions.events.list(session.id).data
            if events and events[-1].type == "session.status_idle":
                break
            time.sleep(1)
        else:
            raise TimeoutError(f"session {session.id} did not idle within 60s")
        agent_events = [e for e in events if e.type == "agent.message"]
        reply = "".join(b.text for e in agent_events for b in e.content)
        return json.loads(reply)
    finally:
        try:
            client.beta.sessions.archive(session.id)
        except Exception:  # noqa: S110
            pass


def score(version: int) -> dict:
    """Evaluate all tickets against the given version and return per-team accuracy."""
    hits = defaultdict(lambda: [0, 0])
    for t in tickets:
        pred = triage(version, t)
        hits[t["team"]][1] += 1
        if pred.get("team") == t["team"]:
            hits[t["team"]][0] += 1
    return dict(hits)


v1_scores = score(version=1)
print("v1 results:")
for team, (correct, total) in sorted(v1_scores.items()):
    print(f"  {team:14s} {correct}/{total}")

v1 results:
  api-platform   5/5
  auth           5/5
  billing        4/5
  dashboard      5/5


模型表现良好，几乎全部路由都正确。

## 发布版本 2

现在这位产品经理发布她的改动：一条路由规则，告诉智能体凡是涉及 API 用量或速率限制的内容都归属平台团队。`agents.update` 把智能体更新为新的系统提示词。


In [6]:
V2_SYSTEM = V1_SYSTEM + (
    "\n\nROUTING RULE: If the ticket text mentions API usage, rate limits, quotas, "
    "or request volume, route to api-platform. Apply this rule before any other "
    "consideration; do not second-guess it based on the rest of the ticket."
)
agent = client.beta.agents.update(AGENT_ID, version=agent.version, system=V2_SYSTEM)
print(f"agent {AGENT_ID} now at v{agent.version}")

agent agent_011CZo6t9Cr6Eg5TeFFPWPpi now at v2


### 代码评审去哪儿了？

我们只用一次 API 调用就改动了智能体。在本笔记本中这完全没有经过评审，用于演示没问题，但不是你在生产环境中应有的做法。

`agents.update` 没有内置的审批工作流。工作区中的任何密钥都能调用它。如果调用方传入的是裸的智能体 ID 而不是固定的版本，它们会在下一个会话就开始使用新提示词。这就是免去部署所付出的代价，与使用特性开关、或者通过 API 而非代码管理任何配置时所做的取舍相同。

把评审环节重新引入的模式是：生产调用方始终固定到某个明确版本，而*那个被固定的编号*才是受变更控制的对象。任何人都可以创建 v2、v3、v10；这些版本停留在服务端，没有流量。晋升意味着更新告诉生产调用方该传入哪个版本的配置，而这次更新要走你正常的评审流程。这个流程让创建版本变得廉价，同时保持你的 SDLC 完整。它仍然是优势，因为无需更新生产代码，你只需更新配置值，改动就会被所有运行器采用。


## 为版本 2 打分

同样的评估，固定到 v2。新规则很宽泛，而在按用量计费的 API 产品上，计费工单也会谈到 API 用量。


In [7]:
v2_scores = score(version=agent.version)

for team in sorted(v1_scores):
    c1, n1 = v1_scores[team]
    c2, n2 = v2_scores[team]
    flag = "  <-- regressed" if c2 < c1 else ""
    print(team)
    print(f"  v1: {c1}/{n1}")
    print(f"  v2: {c2}/{n2}{flag}")

api-platform
  v1: 5/5
  v2: 5/5
auth
  v1: 5/5
  v2: 5/5
billing
  v1: 4/5
  v2: 2/5  <-- regressed
dashboard
  v1: 5/5
  v2: 5/5


## 回滚

计费出现退化。版本 1 仍然存放在服务端。回滚不是一次部署；调用方只需重新传入 `version: 1`。


In [8]:
billing = [t for t in tickets if t["team"] == "billing"]
rerun = [triage(version=1, ticket=t).get("team") for t in billing]
print(f"billing tickets via version=1: {rerun.count('billing')}/{len(billing)} correct")

billing tickets via version=1: 4/5 correct


由于版本存在于服务端，如果这位产品经理想在生产保持 v1 的同时继续迭代 v2，或者把一小部分流量作为金丝雀路由到它，v2 依然在那里。当修复就绪时，她可以创建 v3 并走同样的流程把它晋升。


## 清理


In [9]:
client.beta.agents.archive(AGENT_ID)
client.beta.environments.archive(ENV_ID)
print("archived")

archived


## 小结

这里的机制很小（创建、更新、固定、重新固定），但提示词成为带版本的服务端资源，让你能够独立于应用代码评估和晋升提示词，既可以像上面那样通过 `update` 运行，也可以通过修改配置中的版本。

有几点值得带入你的工作流：

- 让生产调用方固定到某个明确版本，而不是裸的智能体 ID。在你晋升某个版本之前，新版本始终不可见。
- 把被固定的版本号视为改动提示词的关口。创建版本属于探索性质；更新生产环境的固定值才需要走评审。
- 对风险更高的智能体，先把一部分流量路由到新版本并加以对比（正如本笔记本所做），再完全晋升。你实际上可以把这种版本管理当作特性开关使用。

### 下一步

- `client.beta.agents.versions.list(AGENT_ID)` 返回一个智能体的所有版本。
- `managed_agents/` 中的其他笔记本涵盖会话、自定义工具以及端到端模式。
